<a href="https://colab.research.google.com/github/joomha/joomha-CLI/blob/main/joomha_upload_to_hf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📤 Upload Ulang Model ke Hugging Face — `joomha-qwen2.5-coder-3b-repo01`

Notebook ini untuk **menimpa (overwrite)** upload lama yang salah/rusak di Hugging Face.

Alur:
1. Mount Google Drive (tempat hasil training kamu tersimpan)
2. Load model hasil training (adapter LoRA di atas base 4-bit `unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit`) pakai **Unsloth**
3. Merge adapter LoRA ke base model jadi satu model utuh (fp16)
4. Push hasil merge ke Hugging Face, **menimpa** repo lama

## Sebelum mulai:
- **Runtime GPU**: `Runtime > Change runtime type > T4 GPU`
- Siapkan **Hugging Face token** dengan izin **Write** — buat/lihat di https://huggingface.co/settings/tokens
- Pastikan kamu tahu **path folder hasil training di Drive** (isi di cell konfigurasi di bawah)

> 💡 Kenapa pakai Unsloth (bukan `peft` biasa) untuk merge? Karena base model kamu sudah dalam format 4-bit (`bnb-4bit`). Merge LoRA ke base 4-bit itu butuh proses dequantize yang benar — Unsloth punya fungsi bawaan (`push_to_hub_merged`) yang menangani ini dengan aman. Kalau dipaksa merge manual pakai `peft.merge_and_unload()` biasa, hasilnya berisiko korup/kualitas turun — kemungkinan ini juga penyebab upload lama kamu bermasalah.


## 1. Cek GPU

In [1]:
!nvidia-smi

Wed Aug  5 20:40:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive/')


Mounted at /content/drive/


## 3. Install dependencies

Install Unsloth (sudah termasuk transformers, peft, accelerate, bitsandbytes versi yang kompatibel).

In [3]:
!pip install -q -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q -U huggingface_hub


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 123.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 100.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 110.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 21.8 MB/s eta 

## 4. Isi konfigurasi kamu di sini

- `DRIVE_MODEL_PATH` → **isi path folder hasil training di Drive** (yang berisi `adapter_config.json`, `adapter_model.safetensors`, dll). Contoh: `/content/drive/MyDrive/hasil_training/checkpoint-500`
- `HF_REPO_ID` → repo tujuan di Hugging Face (yang mau ditimpa)
- `BASE_MODEL_HINT` → cuma catatan/pengingat, tidak dipakai langsung di kode (Unsloth otomatis baca base model dari `adapter_config.json` di folder training kamu)


In [4]:
# ==== ISI BAGIAN INI DENGAN MILIK KAMU SENDIRI ====
DRIVE_MODEL_PATH = "/content/drive/MyDrive/joomha-training/output/joomha-after-repo-01"  # <-- isi path folder hasil training di Drive, misal "/content/drive/MyDrive/.../checkpoint-500"
HF_REPO_ID       = "joomhadi/ft-model-joomha-base-qwen2.5-coder-3b"
BASE_MODEL_HINT  = "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit"  # info saja
MAX_SEQ_LENGTH   = 2048
# ===================================================

assert DRIVE_MODEL_PATH, "DRIVE_MODEL_PATH masih kosong, isi dulu path folder hasil training di Drive kamu."


## 5. Cek isi folder training (sanity check)

Pastikan folder ini benar-benar berisi checkpoint LoRA (biasanya ada `adapter_config.json` + `adapter_model.safetensors`).

In [5]:
import os

print(f"Isi folder: {DRIVE_MODEL_PATH}\n")
for f in sorted(os.listdir(DRIVE_MODEL_PATH)):
    print(" -", f)

has_adapter_cfg = os.path.exists(os.path.join(DRIVE_MODEL_PATH, "adapter_config.json"))
print("\nTerdeteksi adapter_config.json:", has_adapter_cfg)
if not has_adapter_cfg:
    print("⚠️  Tidak ada adapter_config.json — kemungkinan ini bukan folder checkpoint LoRA.")
    print("   Cek lagi apakah DRIVE_MODEL_PATH sudah menunjuk ke folder checkpoint yang benar.")


Isi folder: /content/drive/MyDrive/joomha-training/output/joomha-after-repo-01

 - README.md
 - adapter_config.json
 - chat_template.jinja
 - log_history.json
 - loss_repo_01.png
 - tokenizer_config.json
 - training_progress.json

Terdeteksi adapter_config.json: True


## 6. Login ke Hugging Face

Dua opsi (pilih salah satu cara jalan cell-nya):
- **Opsi A (disarankan):** simpan token sebagai **Colab Secret** bernama `HF_TOKEN` (ikon 🔑 di sidebar kiri Colab), lalu cell ini otomatis mengambilnya.
- **Opsi B:** kalau secret tidak ditemukan, cell ini akan minta kamu paste token secara manual (tersembunyi, tidak tercetak/tersimpan di notebook).


In [6]:
from huggingface_hub import login

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if not hf_token:
    from getpass import getpass
    print("Colab Secret 'HF_TOKEN' tidak ditemukan. Masukkan token manual (input tersembunyi):")
    hf_token = getpass("Hugging Face token (write access): ")

login(token=hf_token)
print("✅ Berhasil login ke Hugging Face.")


✅ Berhasil login ke Hugging Face.


## 7. Load model hasil training (adapter LoRA + base 4-bit) via Unsloth

Unsloth otomatis membaca `adapter_config.json` di `DRIVE_MODEL_PATH` untuk tahu base model-nya, lalu memuat base + adapter sekaligus.

In [7]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = DRIVE_MODEL_PATH,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,        # auto-detect
    load_in_4bit = True,
)

print("✅ Model & tokenizer berhasil dimuat dari Drive.")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth 2026.8.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


✅ Model & tokenizer berhasil dimuat dari Drive.


## 8. Tes cepat sebelum upload (opsional tapi disarankan)

Pastikan modelnya benar-benar bisa generate teks dengan wajar sebelum kamu timpa repo lama.

In [9]:
FastLanguageModel.for_inference(model)  # aktifkan mode inference cepat (native 2x faster)

test_prompt = "Tulis fungsi Python untuk mengecek apakah sebuah angka termasuk bilangan prima. Jangan lupa menambah docstring pada fungsi Anda."

messages = [{"role": "user", "content": test_prompt}]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

output = model.generate(
    inputs,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.7,
    pad_token_id=tokenizer.eos_token_id,
)

# Ambil cuma bagian jawaban (buang bagian prompt/template-nya)
response = tokenizer.decode(output[0][inputs.shape[-1]:], skip_special_tokens=True)
print(response)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Berdasarkan pemahaman dekat terhadap struktur dan alur kode, berikut adalah contoh implementasi fungsi `is_prime` dalam Python:

```python
def is_prime(n: int) -> bool:
    """
    Fungsi ini memverifikasi apakah angka n adalah bilangan prima.
    
    Args:
    n (int): Angka yang akan diperiksa.
    
    Returns:
    bool: True jika n adalah bilangan prima, False jika tidak.
    """
    if n <= 1:
        return False
    if n <= 3:
        return True
    if n % 2 == 0 or n % 3 == 0:
        return False
    i = 5
    while i * i <= n:
        if n % i == 0 or n % (i + 2) == 0:
            return False
        i += 6
    return True



> Kalau hasil di atas terlihat masuk akal (bukan teks acak/rusak), lanjut ke langkah upload. Kalau hasilnya aneh, **jangan lanjut upload** — cek dulu apakah `DRIVE_MODEL_PATH` menunjuk checkpoint yang benar.

## 9. Push model (merged, fp16) ke Hugging Face — **menimpa repo lama**

`save_method="merged_16bit"` menggabungkan adapter LoRA + base jadi satu model fp16 utuh. Ini yang direkomendasikan supaya notebook inference kamu bisa load dengan `AutoModelForCausalLM` biasa (tanpa perlu `peft` sama sekali saat serving).

Opsi `save_method` lain (ganti kalau perlu):
- `"merged_16bit"` → model utuh fp16, paling kompatibel & disarankan (dipakai di bawah)
- `"merged_4bit"` → model utuh tapi tetap terkuantisasi 4-bit (lebih kecil, tapi saat inference wajib load dengan `bitsandbytes` 4-bit)
- `"lora"` → cuma upload adapter LoRA saja (base model didownload terpisah saat inference)


In [10]:
model.push_to_hub_merged(
    HF_REPO_ID,
    tokenizer,
    save_method="merged_16bit",
    token=hf_token,
)

print(f"✅ Selesai! Cek hasilnya di: https://huggingface.co/{HF_REPO_ID}")


config.json:   0%|          | 0.00/764 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in joomhadi/ft-model-joomha-base-qwen2.5-coder-3b/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0002-of-00002.safetensors:   0%|          |  601kB / 1.21GB            



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [03:05<00:00, 92.78s/it]


Unsloth: Merge process complete. Saved to `/content/joomhadi/ft-model-joomha-base-qwen2.5-coder-3b`
✅ Selesai! Cek hasilnya di: https://huggingface.co/joomhadi/ft-model-joomha-base-qwen2.5-coder-3b


## 10. Setelah upload selesai

- Buka `https://huggingface.co/<HF_REPO_ID>` dan pastikan file yang ter-upload adalah `config.json` + `model.safetensors` (model utuh), **bukan** `adapter_config.json` + `adapter_model.safetensors`.
- Kalau sudah benar, notebook inference kamu (`joomha_coder_inference_api_colab.ipynb`) bisa langsung pakai jalur `AutoModelForCausalLM` biasa tanpa perlu fallback ke `peft` lagi.
- Jangan share token Hugging Face kamu ke siapa pun / commit ke repo publik.
